In [ ]:
# @title Complete Water Training Pipeline (Single Cell)
# Runs: Download → Prepare → Train YOLO → Train ChangeFormer → Evaluate → Verify
# Expected time: ~3-4 hours on T4×2

import os, sys, subprocess, json, time
from pathlib import Path

# ==== CONFIG ====
REPO_URL = "https://github.com/YOUR_ORG/impact-platform.git"  # <-- CHANGE THIS
REPO_DIR = "/kaggle/working/impact-platform"
TILES = ["1", "5", "6", "7", "8"]  # 5 tiles for quick training
BATCH_SIZE = 8  # reduce if OOM
EPOCHS_YOLO = 50
EPOCHS_CHANGE = 200

def run(cmd, desc):
    print(f"\n{'='*60}")
    print(f"▶ {desc}")
    print(f"{'='*60}")
    start = time.time()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = time.time() - start
    if result.returncode != 0:
        print(f"❌ FAILED after {elapsed:.0f}s")
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"{desc} failed")
    print(f"✅ DONE in {elapsed:.0f}s")
    return result.stdout

# ==== 1. SETUP ====
print("🔧 Setting up repository...")
if not os.path.exists(REPO_DIR):
    run(f"git clone {REPO_URL} {REPO_DIR}", "Clone repo")
else:
    run(f"git -C {REPO_DIR} pull", "Pull latest")
os.chdir(f"{REPO_DIR}/apps/ml-service")
print(f"Working in: {os.getcwd()}")

# ==== 2. INSTALL DEPS ====
run("pip install -q -r training/requirements_kaggle.txt", "Install requirements")

# ==== 3. DOWNLOAD DATA ====
print("📥 Downloading S1S2-Water tiles...")
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='idomogalla/s1s2_water',
    repo_type='dataset',
    local_dir='data/raw/s1s2_water',
    allow_patterns=[f"{t}/*" for t in TILES],
    local_dir_use_symlinks=False,
    max_workers=4
)
print(f"✅ Downloaded {len(TILES)} tiles")

# ==== 4. PREPARE YOLO FORMAT ====
run("python -m training.data.prepare_yolo water --source s1s2_water", "Prepare YOLO format")

# ==== 5. TRAIN YOLOv8n ====
print(f"🏋️ Training YOLOv8n for {EPOCHS_YOLO} epochs (batch={BATCH_SIZE})...")
# Override batch size in config
import training.config as cfg_module
cfg_module.CFG.water.water_batch = BATCH_SIZE
run(f"python -m training.scripts.train_yolo water --source s1s2_water", f"Train YOLOv8n ({EPOCHS_YOLO} epochs)")

# ==== 6. TRAIN CHANGEFORMER ====
print(f"🏋️ Training ChangeFormer for {EPOCHS_CHANGE} epochs...")
run(f"python -m training.scripts.train_change water --source s1s2_water", f"Train ChangeFormer ({EPOCHS_CHANGE} epochs)")

# ==== 7. EVALUATE ====
run(f"python -m training.scripts.evaluate water --source s1s2_water --output eval/water_eval_v1.0.json", "Evaluate")

# ==== 8. VERIFY WEIGHTS ====
print("\n" + "="*60)
print("📦 VERIFYING WEIGHTS")
print("="*60)
weights_dir = Path("weights")
for w in ["water_yolov8n.pt", "changeformer_water.pt", "yolov8n.pt"]:
    p = weights_dir / w
    if p.exists():
        size = p.stat().st_size / 1e6
        print(f"  ✅ {w}: {size:.1f} MB")
    else:
        print(f"  ❌ {w}: MISSING")

# ==== 9. QUICK INFERENCE TEST ====
print("\n🧪 QUICK INFERENCE TEST")
try:
    from ultralytics import YOLO
    model = YOLO("weights/water_yolov8n.pt")
    print("✅ YOLO model loads successfully")
except Exception as e:
    print(f"❌ YOLO load failed: {e}")

try:
    import torch
    from training.scripts.train_change import ChangeFormer
    model = ChangeFormer()
    model.load_state_dict(torch.load("weights/changeformer_water.pt"))
    model.eval()
    print("✅ ChangeFormer loads successfully")
except Exception as e:
    print(f"❌ ChangeFormer load failed: {e}")

print("\n🎉 PIPELINE COMPLETE!")
print("Next: python -m training.scripts.promote_model water --version v1.0 --source s1s2_water")
